In [1]:
import numpy as np
import pandas as pd

from SymbolicDSGE import DSGESolver, ModelParser, Shock, SolvedModel
from SymbolicDSGE.monte_carlo import (
    MCPipeline,
    MCData,
    MCContext,
    numpy_operation,
    pandas_operation,
)
from SymbolicDSGE.monte_carlo.operations.core import (
    reference_filter_step,
    simulation_step,
)
from SymbolicDSGE.monte_carlo.operations.regressions import regression_step
from SymbolicDSGE.monte_carlo.operations.tests import (
    ljung_box_test_step,
    wald_test_step,
)
from SymbolicDSGE.monte_carlo.operations.postproc import postproc_step, kde_step
from SymbolicDSGE.monte_carlo.operations.transforms import (
    transform_step,
    standardize_step,
)
import cProfile

In [2]:
model, kalman = ModelParser("../../MODELS/misspec_test/reference.yaml").get_all()

solver = DSGESolver(model, kalman)
compiled = solver.compile(
    n_state=3,
    n_exog=3,
)

steady_state = np.zeros(5, dtype=np.float64)
reference = solver.solve(compiled, steady_state=steady_state)

dgp_model, dgp_kalman = ModelParser("../../MODELS/misspec_test/misspec.yaml").get_all()
dgp_comp = DSGESolver(dgp_model, dgp_kalman).compile(n_state=3, n_exog=3)
dgp_sol = DSGESolver(dgp_model, dgp_kalman)
dgp = dgp_sol.solve(dgp_comp, steady_state=steady_state)

In [3]:
T = 200
n_obs = len(reference.compiled.observable_names)


@numpy_operation
def custom_standardize(
    *, context: MCContext, reference: SolvedModel, dgp: SolvedModel, rep_idx: int
):
    del reference, dgp, rep_idx
    data: MCData = context.require_data()
    obs = data.observables
    if obs is not None:
        return (obs - obs.mean(axis=0)) / obs.std(axis=0)
    return None


@pandas_operation
def get_std_obs_mean(*, traces):
    return traces["payload.custom_std"].mean(axis=0)


pipeline = MCPipeline(
    per_rep_steps=[
        simulation_step(
            T=T,
            target="dgp",
            seed_increment="auto",
            shocks={
                "g,z": Shock(dist="norm", multivar=True, seed=0),
                "r": Shock(dist="norm", seed=1),
            },
            observables=True,
        ),
        reference_filter_step(estimate_R_diag=False),
        transform_step(
            "custom_std",
            func=custom_standardize,
        ),
        standardize_step(
            "builtin_std",
            source="filter",
            field="innov",
        ),
        wald_test_step(
            "std_innov_mean",
            source="filter",
            field="std_innov",
            burn_in=20,
            target=np.zeros(n_obs),
            kind="mean",
        ),
    ],
    postproc_steps=[
        postproc_step(
            "custom_postproc",
            func=get_std_obs_mean,
        ),
        kde_step(
            "builtin_kde",
            trace="payload.builtin_std",
            grid_points=100,
        ),
    ],
)

In [7]:
mc = pipeline.run(
    reference=reference,
    dgp=dgp,
    n_rep=1000,
    retain_payloads=False,
    retain_test_results=False,
    verbosity=2,
)

MC run concluded successfully in 0.75s with 1327.60 it/s.
Per-step Report:

	datagen: 0 faliures, 3361.36 it/s (0.30s).
	filter: 0 faliures, 3453.10 it/s (0.29s).
	custom_std: 0 faliures, 17923.30 it/s (0.06s).
	builtin_std: 0 faliures, 19094.28 it/s (0.05s).
	std_innov_mean: 0 faliures, 22524.86 it/s (0.04s).

Post-processing Report:

	custom_postproc: Succeeded in 0.0007s.
	builtin_kde: Succeeded in 1.3494s.


In [5]:
t_summary = pd.DataFrame(
    {
        name: {
            "mean_statistic": res.mean_statistic,
            "mean_pval": res.mean_pval,
            "rejection_rate": res.rejection_rate,
            "ci_low": res.pval_confidence_interval()[0],
            "ci_high": res.pval_confidence_interval()[1],
        }
        for name, res in mc.test_summaries.items()
    }
).T
print(t_summary.round(3))

                mean_statistic  mean_pval  rejection_rate  ci_low  ci_high
std_innov_mean           2.698      0.555           0.053   0.041    0.069
